# Spotify MPD Fair Benchmark with Sequential, SSL, Efficient, and Multi-Interest Models

This notebook is the current main benchmark for the AML/IRRT project on the Spotify Million Playlist Dataset (MPD).

Its goal is to provide a fair and reproducible comparison between:
- strong sequential recommendation baselines,
- stronger SSL-based sequential learning,
- efficient long-sequence alternatives,
- and a sharper multi-interest architecture with target-aware training.

## Included models

- TopPop
- GRU4Rec
- SASRec
- SASRec + CL4SRec-style SSL
- SASRec + SimDCL-style SSL
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec
- ComiRec-SA-style SASRec with label-aware routing and diversity regularization

## Main benchmark properties

- all neural models are evaluated on the same fixed validation and test playlists
- all models are evaluated over the same filtered item universe
- full-ranking evaluation is implemented in a memory-safe chunked form
- results are aggregated over multiple seeds
- Transformer-family models use causal masking and padding masks
- the notebook uses a stratified playlist subsample
- the analysis focuses on:
  - global ranking quality
  - coverage
  - length-based slices
  - metadata-aware heterogeneity slices
  - target popularity slices
  - unseen-target and hard-continuation slices
  - interest-usage diagnostics for multi-interest models

## Music-aware metadata used in this notebook

This benchmark parses MPD metadata beyond item IDs:
- playlist titles
- track names
- artist names
- album names
- artist and album identifiers
- duration buckets

At the current stage, these signals are mainly used for metadata-aware analysis and slicing. They also prepare later music-aware model extensions.

## Scope note

This notebook focuses on the core benchmark claim:

> whether a stronger and more faithful multi-interest model helps especially on heterogeneous and long-tail playlist continuation.

Heavy generalization diagnostics are intentionally kept out of the main benchmark path.

## 1. Imports and experiment configuration

In [ ]:
import os
import json
import math
import random
import zipfile
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from time import perf_counter
from tqdm.auto import tqdm
from IPython.display import clear_output, display


SPOTIFY_DIR: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd
MPD_MAX_SLICES: None
Filtering: {'MIN_PLAYLIST_LENGTH': 10, 'MAX_PLAYLISTS': None, 'MAX_ITEMS': 10000, 'MAX_SEQ_LEN': 100}
Subsample: {'SUBSAMPLE_FRACTION': 0.7, 'SUBSAMPLE_RANDOM_STATE': 2026}
Model: {'D_MODEL': 128, 'N_HEADS': 4, 'N_LAYERS': 2, 'DROPOUT': 0.1, 'GRU_HIDDEN': 128, 'N_INTERESTS': 4, 'ROUTING_ITERS': 3}
Train: {'BATCH_SIZE': 128, 'LR': 0.001, 'WEIGHT_DECAY': 0.0001, 'EPOCHS': 30, 'PATIENCE': 3}
SSL: {'DO_SSL': True, 'SSL_EPOCHS': 10, 'SSL_TAU': 0.2, 'DEBIAS_INFO_NCE': True}
Eval: {'TOPK': 20, 'EVAL_BATCH_SIZE': 128, 'ITEM_CHUNK_SIZE': 4096, 'NUM_EVAL_PLAYLISTS': None, 'SEEDS': [42, 43]}
DEVICE: cuda


In [ ]:
PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
SPOTIFY_DIR = DATA_DIR / "spotify_mpd"
SPOTIFY_DIR.mkdir(parents=True, exist_ok=True)

MPD_ZIP_PATH = None

MPD_MAX_SLICES = os.environ.get("MPD_MAX_SLICES", "None")
MPD_MAX_SLICES = None if MPD_MAX_SLICES.lower() == "none" else int(MPD_MAX_SLICES)

MIN_PLAYLIST_LENGTH = int(os.environ.get("MIN_PLAYLIST_LENGTH", "10"))
MAX_PLAYLISTS = os.environ.get("MAX_PLAYLISTS", "None")
MAX_PLAYLISTS = None if MAX_PLAYLISTS.lower() == "none" else int(MAX_PLAYLISTS)

MAX_ITEMS = os.environ.get("MAX_ITEMS", "10000")
MAX_ITEMS = None if MAX_ITEMS.lower() == "none" else int(MAX_ITEMS)
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "100"))

SUBSAMPLE_FRACTION = float(os.environ.get("SUBSAMPLE_FRACTION", "0.70"))
SUBSAMPLE_RANDOM_STATE = int(os.environ.get("SUBSAMPLE_RANDOM_STATE", "2026"))

D_MODEL = int(os.environ.get("D_MODEL", "128"))
N_HEADS = int(os.environ.get("N_HEADS", "4"))
N_LAYERS = int(os.environ.get("N_LAYERS", "2"))
DROPOUT = float(os.environ.get("DROPOUT", "0.1"))
GRU_HIDDEN = int(os.environ.get("GRU_HIDDEN", str(D_MODEL)))
N_INTERESTS = int(os.environ.get("N_INTERESTS", "4"))

FMLP_N_LAYERS = int(os.environ.get("FMLP_N_LAYERS", "2"))
FMLP_DROPOUT = float(os.environ.get("FMLP_DROPOUT", str(DROPOUT)))

BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "128"))
LR = float(os.environ.get("LR", "1e-3"))
WEIGHT_DECAY = float(os.environ.get("WEIGHT_DECAY", "1e-4"))
EPOCHS = int(os.environ.get("EPOCHS", "30"))
PATIENCE = int(os.environ.get("PATIENCE", "3"))

DO_CL4SREC_SSL = os.environ.get("DO_CL4SREC_SSL", "1") == "1"
DO_SIMDCL = os.environ.get("DO_SIMDCL", "1") == "1"
SSL_EPOCHS = int(os.environ.get("SSL_EPOCHS", "10"))
SSL_TAU = float(os.environ.get("SSL_TAU", "0.2"))
SIMDCL_NOISE_STD = float(os.environ.get("SIMDCL_NOISE_STD", "0.1"))

COMIREC_LABEL_AWARE = os.environ.get("COMIREC_LABEL_AWARE", "1") == "1"
COMIREC_DIVERSITY_REG = float(os.environ.get("COMIREC_DIVERSITY_REG", "0.05"))
COMIREC_LABEL_TAU = float(os.environ.get("COMIREC_LABEL_TAU", "1.0"))

USE_TITLE_TOKENS = os.environ.get("USE_TITLE_TOKENS", "1") == "1"
USE_METADATA_FEATURES = os.environ.get("USE_METADATA_FEATURES", "1") == "1"

TOPK = int(os.environ.get("TOPK", "20"))
EVAL_BATCH_SIZE = int(os.environ.get("EVAL_BATCH_SIZE", "128"))
ITEM_CHUNK_SIZE = int(os.environ.get("ITEM_CHUNK_SIZE", "4096"))

NUM_EVAL_PLAYLISTS = os.environ.get("NUM_EVAL_PLAYLISTS", "None")
NUM_EVAL_PLAYLISTS = None if NUM_EVAL_PLAYLISTS.lower() == "none" else int(NUM_EVAL_PLAYLISTS)

SEEDS = [42, 43]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("SPOTIFY_DIR:", SPOTIFY_DIR)
print("MPD_MAX_SLICES:", MPD_MAX_SLICES)
print("Filtering:", dict(
    MIN_PLAYLIST_LENGTH=MIN_PLAYLIST_LENGTH,
    MAX_PLAYLISTS=MAX_PLAYLISTS,
    MAX_ITEMS=MAX_ITEMS,
    MAX_SEQ_LEN=MAX_SEQ_LEN,
))
print("Subsample:", dict(
    SUBSAMPLE_FRACTION=SUBSAMPLE_FRACTION,
    SUBSAMPLE_RANDOM_STATE=SUBSAMPLE_RANDOM_STATE,
))
print("Model:", dict(
    D_MODEL=D_MODEL,
    N_HEADS=N_HEADS,
    N_LAYERS=N_LAYERS,
    DROPOUT=DROPOUT,
    GRU_HIDDEN=GRU_HIDDEN,
    N_INTERESTS=N_INTERESTS,
    FMLP_N_LAYERS=FMLP_N_LAYERS,
))
print("Train:", dict(
    BATCH_SIZE=BATCH_SIZE,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EPOCHS=EPOCHS,
    PATIENCE=PATIENCE,
))
print("SSL:", dict(
    DO_CL4SREC_SSL=DO_CL4SREC_SSL,
    DO_SIMDCL=DO_SIMDCL,
    SSL_EPOCHS=SSL_EPOCHS,
    SSL_TAU=SSL_TAU,
    SIMDCL_NOISE_STD=SIMDCL_NOISE_STD,
))
print("ComiRec:", dict(
    COMIREC_LABEL_AWARE=COMIREC_LABEL_AWARE,
    COMIREC_DIVERSITY_REG=COMIREC_DIVERSITY_REG,
    COMIREC_LABEL_TAU=COMIREC_LABEL_TAU,
))
print("Metadata:", dict(
    USE_TITLE_TOKENS=USE_TITLE_TOKENS,
    USE_METADATA_FEATURES=USE_METADATA_FEATURES,
))
print("Eval:", dict(
    TOPK=TOPK,
    EVAL_BATCH_SIZE=EVAL_BATCH_SIZE,
    ITEM_CHUNK_SIZE=ITEM_CHUNK_SIZE,
    NUM_EVAL_PLAYLISTS=NUM_EVAL_PLAYLISTS,
    SEEDS=SEEDS,
))
print("DEVICE:", DEVICE)

## 2. Dataset discovery and raw MPD loading

In this section, the notebook:
- extracts the local MPD archive,
- locates MPD slice files,
- checks that the dataset is available,
- and prepares the raw slice list for parsing.

In [ ]:
def extract_local_zip(zip_path: Optional[Path], out_dir: Path) -> None:
    if zip_path is None:
        print("MPD_ZIP_PATH is None (assuming the dataset is already extracted).")
        return
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip archive not found: {zip_path}")
    print(f"Extracting {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    print("Extraction complete.")

extract_local_zip(MPD_ZIP_PATH, SPOTIFY_DIR)

slice_files = sorted(SPOTIFY_DIR.rglob("mpd.slice.*.json"))
if MPD_MAX_SLICES is not None:
    slice_files = slice_files[:MPD_MAX_SLICES]

print("Found slice files:", len(slice_files))
if len(slice_files) == 0:
    raise FileNotFoundError(
        "No MPD slice files were found. Put extracted files into data/spotify_mpd/ "
        "so that files like 'mpd.slice.0-999.json' are visible."
    )
print("First slice:", slice_files[0])
if len(slice_files) > 1:
    print("Last slice:", slice_files[-1])

MPD_ZIP_PATH is None -> assuming the dataset is already extracted.
Found slice files: 1000
First slice: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd\mpd.slice.0-999.json
Last slice: C:\Users\sokos\DataspellProjects\MML_Project\data\spotify_mpd\mpd.slice.999000-999999.json


## 3. Parse playlist-track interactions

Here I convert MPD slices into a simple interaction table with:
- playlist ID,
- track ID,
- and track position inside the playlist.

This is the raw sequential interaction format used by the rest of the pipeline.

In [ ]:
def safe_str(x) -> str:
    return "" if x is None else str(x)


def duration_bucket(ms: int) -> str:
    if ms <= 0:
        return "unk"
    sec = ms / 1000.0
    if sec < 60:
        return "<1m"
    if sec < 120:
        return "1-2m"
    if sec < 180:
        return "2-3m"
    if sec < 240:
        return "3-4m"
    if sec < 300:
        return "4-5m"
    return "5m+"


def parse_mpd_slices_to_sequences(files: List[Path]) -> pd.DataFrame:
    rows = []
    for path in files:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for pl in data.get("playlists", []):
            pid = int(pl["pid"])
            playlist_name = safe_str(pl.get("name")).strip().lower()

            for pos, tr in enumerate(pl.get("tracks", [])):
                track_uri = tr.get("track_uri")
                if track_uri is None:
                    continue

                duration_ms = int(tr.get("duration_ms", 0) or 0)

                rows.append({
                    "pid": pid,
                    "playlist_name": playlist_name,
                    "track_uri": safe_str(track_uri),
                    "track_name": safe_str(tr.get("track_name")).strip().lower(),
                    "artist_name": safe_str(tr.get("artist_name")).strip().lower(),
                    "album_name": safe_str(tr.get("album_name")).strip().lower(),
                    "artist_uri": safe_str(tr.get("artist_uri")),
                    "album_uri": safe_str(tr.get("album_uri")),
                    "duration_ms": duration_ms,
                    "duration_bucket": duration_bucket(duration_ms),
                    "pos": int(pos),
                })
    return pd.DataFrame(rows)

interactions = parse_mpd_slices_to_sequences(slice_files)
print("Parsed interactions:", interactions.shape)
display(interactions.head())

Parsed interactions: (66346428, 3)


,pid,track_uri,pos
0,0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,0
1,0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,1
2,0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,2
3,0,spotify:track:1AWQoqb9bSvzTjaLralEkT,3
4,0,spotify:track:1lzr43nnXAijIGYnCT8M8H,4


## 4. Interaction cleaning and dataset filtering

This stage:
- sorts interactions by playlist and position,
- removes exact duplicates,
- filters out very short playlists,
- optionally caps the number of playlists and items,
- and applies a stratified subsample by playlist length.

In [ ]:
interactions = interactions.copy()
interactions["pid"] = interactions["pid"].astype(int)
interactions["track_uri"] = interactions["track_uri"].astype(str)
interactions["playlist_name"] = interactions["playlist_name"].fillna("").astype(str)
interactions["track_name"] = interactions["track_name"].fillna("").astype(str)
interactions["artist_name"] = interactions["artist_name"].fillna("").astype(str)
interactions["album_name"] = interactions["album_name"].fillna("").astype(str)
interactions["artist_uri"] = interactions["artist_uri"].fillna("").astype(str)
interactions["album_uri"] = interactions["album_uri"].fillna("").astype(str)
interactions["duration_ms"] = pd.to_numeric(interactions["duration_ms"], errors="coerce").fillna(0).astype(int)
interactions["duration_bucket"] = interactions["duration_bucket"].fillna("unk").astype(str)
interactions["pos"] = interactions["pos"].astype(int)

interactions = (
    interactions
    .sort_values(["pid", "pos"])
    .drop_duplicates(subset=["pid", "track_uri", "pos"])
    .reset_index(drop=True)
)
print("Cleaned interactions:", interactions.shape)
display(interactions.head())

Cleaned interactions: (66346428, 3)


,pid,track_uri,pos
0,0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,0
1,0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,1
2,0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,2
3,0,spotify:track:1AWQoqb9bSvzTjaLralEkT,3
4,0,spotify:track:1lzr43nnXAijIGYnCT8M8H,4


In [5]:
def filter_playlists_by_min_length(df: pd.DataFrame, min_len: int) -> pd.DataFrame:
    cnt = df.groupby("pid").size()
    keep = cnt[cnt >= min_len].index
    return df[df["pid"].isin(keep)].copy()


def cap_top_playlists(df: pd.DataFrame, max_playlists: Optional[int]) -> pd.DataFrame:
    if max_playlists is None or df["pid"].nunique() <= max_playlists:
        return df
    top_playlists = (
        df.groupby("pid")
        .size()
        .sort_values(ascending=False)
        .head(max_playlists)
        .index
    )
    return df[df["pid"].isin(top_playlists)].copy()


def cap_top_items(df: pd.DataFrame, max_items: Optional[int]) -> pd.DataFrame:
    if max_items is None or df["track_uri"].nunique() <= max_items:
        return df
    top_items = (
        df.groupby("track_uri")
        .size()
        .sort_values(ascending=False)
        .head(max_items)
        .index
    )
    return df[df["track_uri"].isin(top_items)].copy()


def build_playlist_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = (
        df.groupby("pid")
        .size()
        .rename("playlist_len")
        .reset_index()
    )

    bins = [9, 19, 39, 79, 199, np.inf]
    labels = ["10-19", "20-39", "40-79", "80-199", "200+"]

    meta["len_bin"] = pd.cut(
        meta["playlist_len"],
        bins=bins,
        labels=labels,
        right=True,
    )
    return meta


def add_length_bin(df: pd.DataFrame) -> pd.DataFrame:
    meta = build_playlist_meta(df)
    out = df.merge(meta, on="pid", how="left")
    return out


def stratified_playlist_subsample(
    df: pd.DataFrame,
    fraction: float,
    random_state: int,
) -> pd.DataFrame:
    meta = build_playlist_meta(df)

    sampled_pids: List[int] = []
    rng = np.random.RandomState(random_state)

    for _, g in meta.groupby("len_bin", observed=False):
        if len(g) == 0:
            continue

        n_take = max(1, int(round(len(g) * fraction)))
        n_take = min(n_take, len(g))

        sampled = g.sample(
            n=n_take,
            random_state=int(rng.randint(0, 10**9)),
        )["pid"].tolist()

        sampled_pids.extend(sampled)

    sampled_pids = sorted(set(sampled_pids))
    return df[df["pid"].isin(sampled_pids)].copy()

In [6]:
df = interactions.copy()
df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)
df = cap_top_playlists(df, MAX_PLAYLISTS)
df = cap_top_items(df, MAX_ITEMS)
df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)

full_meta = build_playlist_meta(df)
full_playlist_meta = (
    full_meta.groupby("len_bin", observed=False)
    .size()
    .rename("n_full")
    .reset_index()
)

df = stratified_playlist_subsample(df, SUBSAMPLE_FRACTION, SUBSAMPLE_RANDOM_STATE)
df = filter_playlists_by_min_length(df, MIN_PLAYLIST_LENGTH)
df = add_length_bin(df)
df = df.sort_values(["pid", "pos"]).reset_index(drop=True)

sub_meta = build_playlist_meta(df)
sub_playlist_meta = (
    sub_meta.groupby("len_bin", observed=False)
    .size()
    .rename("n_subsample")
    .reset_index()
)

display(
    full_playlist_meta
    .merge(sub_playlist_meta, on="len_bin", how="outer")
    .fillna(0)
)

print("Interactions after stratified subsample:", df.shape)
print("Unique playlists after subsample:", df["pid"].nunique())
print("Unique tracks after subsample:", df["track_uri"].nunique())

,len_bin,n_full,n_subsample
0,10-19,184983,129488
1,20-39,237617,166332
2,40-79,207874,145512
3,80-199,120973,84681
4,200+,1999,1399


Interactions after stratified subsample: (24716682, 5)
Unique playlists after subsample: 527412
Unique tracks after subsample: 10000


## 5. Remap playlists and tracks to integer IDs

The models operate on compact integer IDs rather than raw playlist or track identifiers.

This section:
- maps playlists to user IDs,
- maps tracks to item IDs,
- and prepares the final interaction table used for sequence modeling.

In [7]:
def remap_series_to_int(s: pd.Series) -> Tuple[pd.Series, Dict[str, int]]:
    uniq = s.unique()
    mapping = {v: i + 1 for i, v in enumerate(uniq)}
    return s.map(mapping).astype(np.int32), mapping


df["user_id"], playlist_map = remap_series_to_int(df["pid"].astype(str))
df["item_id"], item_map = remap_series_to_int(df["track_uri"])
df = df.drop(columns=["track_uri"])

n_users = int(df["user_id"].max())
n_items = int(df["item_id"].max())

print("Filtered interactions:", df.shape)
print("Playlists:", n_users, "| Tracks:", n_items)
display(df.head())

Filtered interactions: (24716682, 6)
Playlists: 527412 | Tracks: 10000


,pid,pos,playlist_len,len_bin,user_id,item_id
0,3,23,13,10-19,1,1
1,3,26,13,10-19,1,2
2,3,28,13,10-19,1,3
3,3,31,13,10-19,1,4
4,3,32,13,10-19,1,5


In [8]:
item_freq = df["item_id"].value_counts()
print("Most frequent item_id:", int(item_freq.index[0]), "count:", int(item_freq.iloc[0]))
print("Unique items:", df["item_id"].nunique())

Most frequent item_id: 328 count: 32167
Unique items: 10000


## 5.1 Build item and playlist metadata tables

In this section, I derive compact metadata tables that can be used for:
- metadata-aware slicing,
- music-aware diversity analysis,
- artist / album based sequence features,
- and later metadata-aware model extensions.

In [ ]:
item_meta = (
    df.groupby("item_id", as_index=False)
    .agg(
        track_name=("track_name", "first"),
        artist_name=("artist_name", "first"),
        album_name=("album_name", "first"),
        artist_uri=("artist_uri", "first"),
        album_uri=("album_uri", "first"),
        duration_bucket=("duration_bucket", "first"),
    )
)

playlist_meta = (
    df.groupby("user_id", as_index=False)
    .agg(
        playlist_name=("playlist_name", "first"),
    )
)

itemid_to_artist = dict(zip(item_meta["item_id"], item_meta["artist_uri"]))
itemid_to_album = dict(zip(item_meta["item_id"], item_meta["album_uri"]))
itemid_to_duration_bucket = dict(zip(item_meta["item_id"], item_meta["duration_bucket"]))
itemid_to_track_name = dict(zip(item_meta["item_id"], item_meta["track_name"]))
itemid_to_artist_name = dict(zip(item_meta["item_id"], item_meta["artist_name"]))
itemid_to_album_name = dict(zip(item_meta["item_id"], item_meta["album_name"]))

userid_to_playlist_title = dict(zip(playlist_meta["user_id"], playlist_meta["playlist_name"]))

display(item_meta.head())
display(playlist_meta.head())

## 6. Build sequential train/validation/test splits

For each playlist, I build an ordered item sequence and use a simple last-item protocol:
- train sequence: all items except the last two,
- validation sequence: all items except the last one,
- test sequence: full sequence with the last item as the target.

This keeps the setup consistent with next-item prediction.

In [9]:
@dataclass
class SeqData:
    user_seqs: Dict[int, List[int]]
    n_users: int
    n_items: int


def left_pad(seq: List[int], max_len: int, pad: int = 0) -> List[int]:
    if len(seq) >= max_len:
        return seq[-max_len:]
    return [pad] * (max_len - len(seq)) + seq


user_seqs = df.groupby("user_id")["item_id"].apply(list).to_dict()
seqdata = SeqData(user_seqs=user_seqs, n_users=n_users, n_items=n_items)


def split_train_valid_test(seqdata: SeqData):
    train_seqs, valid_seqs, test_seqs = {}, {}, {}
    for u, seq in seqdata.user_seqs.items():
        if len(seq) < 3:
            continue
        train_seqs[u] = seq[:-2]
        valid_seqs[u] = seq[:-1]
        test_seqs[u] = seq[:]
    return train_seqs, valid_seqs, test_seqs


train_seqs, valid_seqs, test_seqs = split_train_valid_test(seqdata)
print("Playlists with train/valid/test sequences:", len(train_seqs))

Playlists with train/valid/test sequences: 527412


## 7. Freeze evaluation users and evaluation cases

To make model comparison fair, I freeze:
- validation playlists,
- test playlists,
- and the exact sequence histories used during evaluation.

This ensures that all models are scored on the same cases.

In [10]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_fixed_eval_users(eval_seqs: Dict[int, List[int]], num_eval_users: Optional[int], seed: int) -> List[int]:
    users = list(eval_seqs.keys())
    rng = random.Random(seed)
    rng.shuffle(users)
    if num_eval_users is None:
        return users
    return users[:min(num_eval_users, len(users))]


@dataclass
class EvalCases:
    users: List[int]
    histories: np.ndarray
    targets: np.ndarray


def build_eval_cases(
    eval_seqs: Dict[int, List[int]],
    users: List[int],
    max_len: int,
) -> EvalCases:
    histories, targets = [], []

    for u in users:
        seq = eval_seqs[u]
        hist = left_pad(seq[:-1], max_len)
        target = seq[-1]

        histories.append(hist)
        targets.append(target)

    return EvalCases(
        users=users,
        histories=np.asarray(histories, dtype=np.int64),
        targets=np.asarray(targets, dtype=np.int64),
    )


EVAL_USER_SEED = 2026

valid_users_fixed = build_fixed_eval_users(valid_seqs, NUM_EVAL_PLAYLISTS, EVAL_USER_SEED)
test_users_fixed = build_fixed_eval_users(test_seqs, NUM_EVAL_PLAYLISTS, EVAL_USER_SEED)

valid_cases = build_eval_cases(valid_seqs, valid_users_fixed, MAX_SEQ_LEN)
test_cases = build_eval_cases(test_seqs, test_users_fixed, MAX_SEQ_LEN)

print("Validation cases:", len(valid_cases.users))
print("Test cases:", len(test_cases.users))

Validation cases: 527412
Test cases: 527412


## 8. Build analysis metadata and subset definitions

Besides global test metrics, the notebook evaluates targeted subsets that are consistent with the no-repeat playlist continuation setting.

This section creates:
- length-based slices,
- metadata-aware heterogeneity slices,
- target popularity slices,
- unseen-target slices,
- and hard-continuation slices.

In [ ]:
def safe_norm_entropy_from_counts(counts: np.ndarray) -> float:
    counts = counts[counts > 0].astype(np.float64)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    ent = -(p * np.log(p + 1e-12)).sum()
    max_ent = math.log(len(p))
    return 0.0 if max_ent <= 0 else float(ent / max_ent)


def compute_history_diversity_features(
    seq: List[int],
    artist_map: Dict[int, str],
    album_map: Dict[int, str],
    duration_bucket_map: Dict[int, str],
) -> Dict[str, float]:
    hist = list(seq)
    L = len(hist)

    if L == 0:
        return {
            "hist_len": 0.0,
            "item_unique_ratio": 0.0,
            "artist_unique_ratio": 0.0,
            "album_unique_ratio": 0.0,
            "item_entropy": 0.0,
            "artist_entropy": 0.0,
            "duration_entropy": 0.0,
            "diversity_score": 0.0,
        }

    item_counts = np.array(list(Counter(hist).values()), dtype=np.int64)

    artists = [artist_map.get(x, "") for x in hist]
    albums = [album_map.get(x, "") for x in hist]
    durations = [duration_bucket_map.get(x, "unk") for x in hist]

    artist_counts = np.array(list(Counter(artists).values()), dtype=np.int64)
    duration_counts = np.array(list(Counter(durations).values()), dtype=np.int64)

    item_unique_ratio = len(set(hist)) / L
    artist_unique_ratio = len(set(artists)) / L
    album_unique_ratio = len(set(albums)) / L

    item_entropy = safe_norm_entropy_from_counts(item_counts)
    artist_entropy = safe_norm_entropy_from_counts(artist_counts)
    duration_entropy = safe_norm_entropy_from_counts(duration_counts)

    diversity_score = float(
        0.35 * item_unique_ratio
        + 0.35 * artist_entropy
        + 0.15 * album_unique_ratio
        + 0.15 * duration_entropy
    )

    return {
        "hist_len": float(L),
        "item_unique_ratio": float(item_unique_ratio),
        "artist_unique_ratio": float(artist_unique_ratio),
        "album_unique_ratio": float(album_unique_ratio),
        "item_entropy": float(item_entropy),
        "artist_entropy": float(artist_entropy),
        "duration_entropy": float(duration_entropy),
        "diversity_score": diversity_score,
    }


def build_eval_analysis_meta(
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    track_pop: pd.Series,
    artist_map: Dict[int, str],
    album_map: Dict[int, str],
    duration_bucket_map: Dict[int, str],
) -> pd.DataFrame:
    rows = []

    pop_values = track_pop.astype(float)
    tail_thr = float(pop_values.quantile(0.20))
    head_thr = float(pop_values.quantile(0.80))

    for idx, u in enumerate(eval_cases.users):
        full_seq = eval_seqs[u]
        hist = full_seq[:-1]
        target = full_seq[-1]

        f = compute_history_diversity_features(
            seq=hist,
            artist_map=artist_map,
            album_map=album_map,
            duration_bucket_map=duration_bucket_map,
        )

        target_pop = float(track_pop.get(target, 0.0))
        if target_pop <= tail_thr:
            target_pop_bucket = "tail"
        elif target_pop >= head_thr:
            target_pop_bucket = "head"
        else:
            target_pop_bucket = "mid"

        last3 = hist[-3:] if len(hist) >= 3 else hist[:]
        target_unseen = int(target not in hist)
        hard_continuation = int(target not in last3)

        rows.append({
            "row_idx": idx,
            "user_id": u,
            "target": target,
            "target_pop": target_pop,
            "target_pop_bucket": target_pop_bucket,
            "target_unseen": target_unseen,
            "hard_continuation": hard_continuation,
            **f,
        })

    meta = pd.DataFrame(rows)

    q_len_33 = meta["hist_len"].quantile(0.33)
    q_len_67 = meta["hist_len"].quantile(0.67)

    def len_bucket_fn(x: float) -> str:
        if x <= q_len_33:
            return "short"
        if x >= q_len_67:
            return "long"
        return "medium"

    meta["len_bucket"] = meta["hist_len"].map(len_bucket_fn)

    q_div_33 = meta["diversity_score"].quantile(0.33)
    q_div_67 = meta["diversity_score"].quantile(0.67)

    def div_bucket_fn(x: float) -> str:
        if x <= q_div_33:
            return "homogeneous"
        if x >= q_div_67:
            return "heterogeneous"
        return "medium"

    meta["div_bucket"] = meta["diversity_score"].map(div_bucket_fn)
    return meta


def build_subset_masks(test_meta: pd.DataFrame) -> Dict[str, np.ndarray]:
    masks: Dict[str, np.ndarray] = {}
    masks["all"] = np.ones(len(test_meta), dtype=bool)

    for bucket in ["short", "medium", "long"]:
        masks[f"len_{bucket}"] = (test_meta["len_bucket"].values == bucket)

    for bucket in ["homogeneous", "medium", "heterogeneous"]:
        masks[f"div_{bucket}"] = (test_meta["div_bucket"].values == bucket)

    for bucket in ["head", "mid", "tail"]:
        masks[f"target_pop_{bucket}"] = (test_meta["target_pop_bucket"].values == bucket)

    masks["target_unseen"] = test_meta["target_unseen"].values.astype(bool)
    masks["hard_continuation"] = test_meta["hard_continuation"].values.astype(bool)

    return {k: v for k, v in masks.items() if v.sum() > 0}
    


def metrics_at_k(ranked: np.ndarray, target: np.ndarray, k: int):
    hit = (ranked[:, :k] == target[:, None])
    recall = float(hit.any(axis=1).mean())
    first_pos = hit.argmax(axis=1)
    has_hit = hit.any(axis=1)
    rr = np.zeros(hit.shape[0], dtype=np.float64)
    rr[has_hit] = 1.0 / (first_pos[has_hit] + 1.0)
    mrr = float(rr.mean())
    ndcg = np.zeros(hit.shape[0], dtype=np.float64)
    ndcg[has_hit] = 1.0 / np.log2(first_pos[has_hit] + 2.0)
    ndcg = float(ndcg.mean())
    return recall, ndcg, mrr


def coverage_at_k(ranked: np.ndarray, n_items: int, k: int) -> float:
    return float(len(np.unique(ranked[:, :k])) / n_items)


def compute_metrics_from_ranked(
    ranked: np.ndarray,
    targets: np.ndarray,
    n_items: int,
    k: int,
) -> Dict[str, float]:
    r, n, m = metrics_at_k(ranked, targets, k)
    cov = coverage_at_k(ranked, n_items, k)
    return {
        f"Recall@{k}": float(r),
        f"NDCG@{k}": float(n),
        f"MRR@{k}": float(m),
        f"Coverage@{k}": float(cov),
    }


def summarize_subset_metrics_from_ranked(
    ranked: np.ndarray,
    targets: np.ndarray,
    n_items: int,
    k: int,
    subset_masks: Dict[str, np.ndarray],
    model_name: str,
    seed,
) -> List[Dict[str, object]]:
    rows = []
    for subset_name, mask in subset_masks.items():
        ranked_sub = ranked[mask]
        targets_sub = targets[mask]
        if len(targets_sub) == 0:
            continue

        metrics = compute_metrics_from_ranked(
            ranked=ranked_sub,
            targets=targets_sub,
            n_items=n_items,
            k=k,
        )
        rows.append({
            "model": model_name,
            "seed": seed,
            "subset": subset_name,
            "n_cases": int(mask.sum()),
            **metrics,
        })
    return rows


track_pop = df.groupby("item_id").size().rename("track_pop")
test_analysis_meta = build_eval_analysis_meta(
    eval_cases=test_cases,
    eval_seqs=test_seqs,
    track_pop=track_pop,
    artist_map=itemid_to_artist,
    album_map=itemid_to_album,
    duration_bucket_map=itemid_to_duration_bucket,
)
TEST_SUBSET_MASKS = build_subset_masks(test_analysis_meta)

print("Test analysis meta shape:", test_analysis_meta.shape)
display(test_analysis_meta.head())

Test analysis meta shape: (527412, 19)


,row_idx,user_id,target,target_pop,target_pop_bucket,target_in_hist,target_in_last1,target_in_last3,target_old_repeat,target_unseen,hist_len,n_unique_hist,unique_ratio,repeat_ratio,norm_item_entropy,diversity_score,len_bucket,div_bucket,hard_continuation
0,0,225067,330,24234.0,head,0,0,0,0,1,33.0,33.0,1.000000,0.000000,1.000000,1.000000,medium,medium,1
1,1,319036,192,5892.0,head,0,0,0,0,1,112.0,109.0,0.973214,0.026786,0.997872,0.985543,long,homogeneous,1
2,2,325210,697,5621.0,head,0,0,0,0,1,10.0,10.0,1.000000,0.000000,1.000000,1.000000,short,heterogeneous,1
3,3,475831,5800,3001.0,mid,0,0,0,0,1,72.0,69.0,0.958333,0.041667,0.996409,0.977371,long,homogeneous,1
4,4,41287,6088,2731.0,mid,0,0,0,0,1,24.0,24.0,1.000000,0.000000,1.000000,1.000000,medium,heterogeneous,1


In [12]:
print("Available subsets:")
for name, mask in TEST_SUBSET_MASKS.items():
    print(f"{name:>20s}: n={int(mask.sum())}")

Available subsets:
                 all: n=527412
           len_short: n=181217
          len_medium: n=171636
            len_long: n=174559
     div_homogeneous: n=174457
          div_medium: n=174988
   div_heterogeneous: n=177967
     target_pop_head: n=247065
      target_pop_mid: n=231205
     target_pop_tail: n=49142
     target_in_last1: n=2234
     target_in_last3: n=3788
   target_old_repeat: n=10183
       target_unseen: n=513441
   hard_continuation: n=523624


In [13]:
pop = Counter()
for seq in train_seqs.values():
    pop.update(seq)
TOPPOP = [item for item, _ in pop.most_common(TOPK)]
print("TopPop head:", TOPPOP)

TopPop head: [328, 1335, 379, 716, 1411, 1101, 326, 1696, 330, 2400, 1270, 1469, 496, 331, 1660, 463, 146, 1442, 1614, 390]


## 9. Training datasets and SSL augmentations

This section defines:
- pairwise next-item training samples,
- CL4SRec-style sequence augmentations,
- SimDCL-style representation perturbation,
- and contrastive objectives used for SSL pretraining.

In [ ]:
def sample_negative(n_items: int, forbidden: set[int], rng: random.Random) -> int:
    while True:
        j = rng.randint(1, n_items)
        if j not in forbidden:
            return j


class PairwiseNextItemDataset(Dataset):
    def __init__(
        self,
        train_seqs: Dict[int, List[int]],
        max_len: int,
        n_items: int,
        n_neg: int = 50,
        max_prefix_samples_per_user: Optional[int] = None,
        seed: int = 42,
    ):
        self.instances = []
        rng = random.Random(seed)

        for _, seq in train_seqs.items():
            if len(seq) < 2:
                continue

            positions = list(range(1, len(seq)))
            if max_prefix_samples_per_user is not None and len(positions) > max_prefix_samples_per_user:
                positions = rng.sample(positions, max_prefix_samples_per_user)
                positions.sort()

            forbidden = set(seq)
            for t in positions:
                hist = seq[:t]
                pos = seq[t]
                negs = [sample_negative(n_items, forbidden, rng) for _ in range(n_neg)]
                self.instances.append((left_pad(hist, max_len), pos, negs))

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        seq, pos, negs = self.instances[idx]
        return (
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(pos, dtype=torch.long),
            torch.tensor(negs, dtype=torch.long),
        )


class ContrastiveDataset(Dataset):
    def __init__(self, train_seqs: Dict[int, List[int]], max_len: int):
        self.users = list(train_seqs.keys())
        self.train_seqs = train_seqs
        self.max_len = max_len

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        seq = self.train_seqs[u]
        v1 = left_pad(make_view(seq), self.max_len)
        v2 = left_pad(make_view(seq), self.max_len)
        return torch.tensor(v1, dtype=torch.long), torch.tensor(v2, dtype=torch.long)


def aug_crop(seq, min_ratio=0.5):
    if len(seq) < 2:
        return seq
    L = len(seq)
    new_len = max(2, int(L * random.uniform(min_ratio, 1.0)))
    start = random.randint(0, L - new_len)
    return seq[start:start + new_len]


def aug_mask(seq, p=0.2):
    out = seq.copy()
    for i in range(len(out)):
        if random.random() < p:
            out[i] = 0
    return out


def aug_reorder(seq, window=3):
    out = seq.copy()
    if len(out) <= window:
        random.shuffle(out)
        return out
    start = random.randint(0, len(out) - window)
    sub = out[start:start + window]
    random.shuffle(sub)
    out[start:start + window] = sub
    return out


def make_view(seq):
    s = aug_crop(seq)
    s = aug_reorder(s, window=3)
    s = aug_mask(s, p=0.2)
    if len(s) > 0 and all(x == 0 for x in s):
        s[random.randrange(len(s))] = random.choice(seq)
    return s


def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return F.cross_entropy(logits, labels)


def simdcl_loss(z: torch.Tensor, noise_std: float, temperature: float):
    z1 = F.normalize(z, dim=-1)
    z2 = F.normalize(z + torch.randn_like(z) * noise_std, dim=-1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return F.cross_entropy(logits, labels)

Models ready.


## 10. Model architectures

This benchmark includes:
- TopPop
- GRU4Rec
- SASRec
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec

Transformer-family models use both:
- causal attention masks
- padding masks

The multi-interest model uses:
- target-aware label routing during training
- diversity regularization between interest vectors

In [ ]:
def build_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    return torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()


def gather_last_nonpad(hidden: torch.Tensor, seq: torch.Tensor) -> torch.Tensor:
    idx = (seq != 0).sum(dim=1) - 1
    idx = idx.clamp_min(0)
    return hidden[torch.arange(hidden.size(0), device=hidden.device), idx]

In [ ]:
class SequenceModelBase(nn.Module):
    def get_user_repr(self, seq: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def score_items_from_repr(self, user_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def pooled_ssl_repr(self, seq: torch.Tensor) -> torch.Tensor:
        return self.get_user_repr(seq)

In [ ]:
class GRU4Rec(SequenceModelBase):
    def __init__(self, n_items: int, d_model: int, hidden_size: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.proj = nn.Linear(hidden_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(d_model)

    def get_user_repr(self, seq: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.item_emb(seq))
        h, _ = self.gru(x)
        out = gather_last_nonpad(h, seq)
        return self.ln(self.proj(out))

    def score_items_from_repr(self, user_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class SASRec(SequenceModelBase):
    def __init__(self, n_items: int, max_len: int, d_model: int, n_heads: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln = nn.LayerNorm(d_model)

    def forward_hidden(self, seq: torch.Tensor) -> torch.Tensor:
        B, L = seq.shape
        pos = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq) + self.pos_emb(pos)
        x = self.drop(x)

        attn_mask = build_causal_mask(L, seq.device)
        pad_mask = (seq == 0)

        h = self.encoder(
            x,
            mask=attn_mask,
            src_key_padding_mask=pad_mask,
        )
        return self.ln(h)

    def get_user_repr(self, seq: torch.Tensor) -> torch.Tensor:
        h = self.forward_hidden(seq)
        return gather_last_nonpad(h, seq)

    def score_items_from_repr(self, user_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class FMLPBlock(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.filter = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        y = self.filter(self.ln1(x))
        x = x + y
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        z = self.ffn(self.ln2(x))
        x = x + z
        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        return x


class FMLPRec(SequenceModelBase):
    def __init__(self, n_items: int, max_len: int, d_model: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([FMLPBlock(d_model=d_model, dropout=dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)

    def forward_hidden(self, seq: torch.Tensor) -> torch.Tensor:
        B, L = seq.shape
        pos = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq) + self.pos_emb(pos)
        x = self.drop(x)
        pad_mask = (seq == 0)

        x = x.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        for block in self.blocks:
            x = block(x, pad_mask)
        return self.ln(x)

    def get_user_repr(self, seq: torch.Tensor) -> torch.Tensor:
        h = self.forward_hidden(seq)
        return gather_last_nonpad(h, seq)

    def score_items_from_repr(self, user_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_repr.unsqueeze(1)).sum(-1)


class MultiInterestBase(nn.Module):
    def get_interest_repr(self, seq: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def pooled_ssl_repr(self, seq: torch.Tensor) -> torch.Tensor:
        return self.get_interest_repr(seq).mean(dim=1)

    def score_items(self, interest_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class ComiRecSAStyleSASRec(MultiInterestBase):
    def __init__(self, n_items: int, max_len: int, d_model: int, n_heads: int, n_layers: int, dropout: float, n_interests: int):
        super().__init__()
        self.n_interests = n_interests
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln = nn.LayerNorm(d_model)
        self.interest_queries = nn.Parameter(torch.randn(n_interests, d_model) * 0.02)

    def forward_hidden(self, seq: torch.Tensor) -> torch.Tensor:
        B, L = seq.shape
        pos = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq) + self.pos_emb(pos)
        x = self.drop(x)

        attn_mask = build_causal_mask(L, seq.device)
        pad_mask = (seq == 0)

        h = self.encoder(
            x,
            mask=attn_mask,
            src_key_padding_mask=pad_mask,
        )
        return self.ln(h)

    def get_interest_repr(self, seq: torch.Tensor) -> torch.Tensor:
        h = self.forward_hidden(seq)
        B, L, D = h.shape
        mask = (seq != 0)

        q = self.interest_queries.unsqueeze(0).expand(B, self.n_interests, D)
        attn_logits = torch.matmul(q, h.transpose(1, 2)) / math.sqrt(D)
        attn_logits = attn_logits.masked_fill(~mask.unsqueeze(1), -1e9)
        attn = torch.softmax(attn_logits, dim=-1)
        z = torch.matmul(attn, h)
        z = F.normalize(z, dim=-1)
        return z

    def score_items(self, interest_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        scores = torch.einsum("bkd,bcd->bkc", interest_repr, item_vec)
        return scores.max(dim=1).values

    def score_items_per_interest(self, interest_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        return torch.einsum("bkd,bcd->bkc", interest_repr, item_vec)

In [15]:
def predict_toppop_ranked(
    eval_cases: EvalCases,
    train_or_eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    k: int = 20,
) -> np.ndarray:
    item_ids = np.arange(1, n_items + 1, dtype=np.int32)

    global_scores = np.asarray(
        [pop_counter.get(int(i), 0) for i in item_ids],
        dtype=np.int32,
    )

    global_order = np.lexsort((item_ids, -global_scores))
    global_ranked_items = item_ids[global_order]

    ranked = np.zeros((len(eval_cases.users), k), dtype=np.int32)

    for row_idx, u in enumerate(eval_cases.users):
        seen_items = set(train_or_eval_seqs[u][:-1])

        out = []
        for item in global_ranked_items:
            if item not in seen_items:
                out.append(item)
                if len(out) == k:
                    break

        ranked[row_idx, :] = out

    return ranked


def evaluate_toppop_full_ranking(
    eval_cases: EvalCases,
    train_or_eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    k: int = 20,
):
    ranked = predict_toppop_ranked(
        eval_cases=eval_cases,
        train_or_eval_seqs=train_or_eval_seqs,
        pop_counter=pop_counter,
        n_items=n_items,
        k=k,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]

## 11. Chunked full-ranking evaluation

Full ranking over all items is implemented in a chunked form to avoid GPU memory overflow.

This section defines:
- TopPop ranking,
- single-representation ranking,
- multi-interest ranking,
- and shared metric computation utilities.

In [ ]:
@torch.no_grad()
def predict_repr_model_ranked(
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> np.ndarray:
    model.eval()
    ranked_all = []

    all_item_ids_cpu = torch.arange(1, n_items + 1, dtype=torch.long)

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))

        batch_users = eval_cases.users[start:end]
        X = torch.tensor(eval_cases.histories[start:end], dtype=torch.long, device=DEVICE)

        user_repr = model.get_user_repr(X)  # [B, D]
        batch_size = X.size(0)

        seen_sets = [set(eval_seqs[u][:-1]) for u in batch_users]

        batch_top_scores = torch.full(
            (batch_size, k),
            -1e9,
            dtype=torch.float32,
            device=DEVICE,
        )
        batch_top_items = torch.zeros(
            (batch_size, k),
            dtype=torch.long,
            device=DEVICE,
        )

        for item_start in range(0, n_items, item_chunk_size):
            item_end = min(item_start + item_chunk_size, n_items)

            chunk_item_ids = all_item_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_item_emb = model.item_emb(chunk_item_ids)

            scores_chunk = torch.matmul(user_repr, chunk_item_emb.transpose(0, 1))

            for i, seen_items in enumerate(seen_sets):
                local_seen = [
                    item - item_start - 1
                    for item in seen_items
                    if item_start < item <= item_end
                ]
                if local_seen:
                    scores_chunk[i, local_seen] = -1e9

            chunk_k = min(k, scores_chunk.size(1))
            chunk_top_scores, chunk_top_idx = torch.topk(scores_chunk, k=chunk_k, dim=1)
            chunk_top_items = chunk_item_ids[chunk_top_idx]

            merged_scores = torch.cat([batch_top_scores, chunk_top_scores], dim=1)
            merged_items = torch.cat([batch_top_items, chunk_top_items], dim=1)

            new_top_scores, new_top_pos = torch.topk(merged_scores, k=k, dim=1)
            new_top_items = torch.gather(merged_items, 1, new_top_pos)

            batch_top_scores = new_top_scores
            batch_top_items = new_top_items

            del chunk_item_ids, chunk_item_emb, scores_chunk
            del chunk_top_scores, chunk_top_idx, chunk_top_items
            del merged_scores, merged_items, new_top_scores, new_top_pos, new_top_items

            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        ranked_all.append(batch_top_items.cpu().numpy())

        del X, user_repr, batch_top_scores, batch_top_items
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(ranked_all)


@torch.no_grad()
def evaluate_repr_model_full_ranking(
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
):
    ranked = predict_repr_model_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]


@torch.no_grad()
def predict_multi_interest_ranked(
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> np.ndarray:
    model.eval()
    ranked_all = []

    all_item_ids_cpu = torch.arange(1, n_items + 1, dtype=torch.long)

    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))

        batch_users = eval_cases.users[start:end]
        X = torch.tensor(eval_cases.histories[start:end], dtype=torch.long, device=DEVICE)

        z = model.get_interest_repr(X)
        batch_size = X.size(0)

        seen_sets = [set(eval_seqs[u][:-1]) for u in batch_users]

        batch_top_scores = torch.full(
            (batch_size, k),
            -1e9,
            dtype=torch.float32,
            device=DEVICE,
        )
        batch_top_items = torch.zeros(
            (batch_size, k),
            dtype=torch.long,
            device=DEVICE,
        )

        for item_start in range(0, n_items, item_chunk_size):
            item_end = min(item_start + item_chunk_size, n_items)

            chunk_item_ids = all_item_ids_cpu[item_start:item_end].to(DEVICE)
            chunk_item_emb = model.item_emb(chunk_item_ids)

            scores_per_interest = torch.einsum("bkd,cd->bkc", z, chunk_item_emb)
            scores_chunk = scores_per_interest.max(dim=1).values

            for i, seen_items in enumerate(seen_sets):
                local_seen = [
                    item - item_start - 1
                    for item in seen_items
                    if item_start < item <= item_end
                ]
                if local_seen:
                    scores_chunk[i, local_seen] = -1e9

            chunk_k = min(k, scores_chunk.size(1))
            chunk_top_scores, chunk_top_idx = torch.topk(scores_chunk, k=chunk_k, dim=1)
            chunk_top_items = chunk_item_ids[chunk_top_idx]

            merged_scores = torch.cat([batch_top_scores, chunk_top_scores], dim=1)
            merged_items = torch.cat([batch_top_items, chunk_top_items], dim=1)

            new_top_scores, new_top_pos = torch.topk(merged_scores, k=k, dim=1)
            new_top_items = torch.gather(merged_items, 1, new_top_pos)

            batch_top_scores = new_top_scores
            batch_top_items = new_top_items

            del chunk_item_ids, chunk_item_emb, scores_per_interest, scores_chunk
            del chunk_top_scores, chunk_top_idx, chunk_top_items
            del merged_scores, merged_items, new_top_scores, new_top_pos, new_top_items

            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        ranked_all.append(batch_top_items.cpu().numpy())

        del X, z, batch_top_scores, batch_top_items
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(ranked_all)


@torch.no_grad()
def evaluate_multi_interest_full_ranking(
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
):
    ranked = predict_multi_interest_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    metrics = compute_metrics_from_ranked(ranked, eval_cases.targets, n_items, k)
    return metrics[f"Recall@{k}"], metrics[f"NDCG@{k}"], metrics[f"MRR@{k}"], metrics[f"Coverage@{k}"]


@torch.no_grad()
def analyze_interest_usage(model: MultiInterestBase, eval_cases: EvalCases, eval_batch_size: int = 256):
    model.eval()
    all_best_interest, all_interest_entropy = [], []
    for start in range(0, len(eval_cases.users), eval_batch_size):
        end = min(start + eval_batch_size, len(eval_cases.users))
        X = torch.tensor(eval_cases.histories[start:end], dtype=torch.long, device=DEVICE)
        T = torch.tensor(eval_cases.targets[start:end], dtype=torch.long, device=DEVICE).unsqueeze(1)
        z = model.get_interest_repr(X)
        item_vec = model.item_emb(T).squeeze(1)
        interest_scores = torch.einsum("bkd,bd->bk", z, item_vec)
        best_interest = interest_scores.argmax(dim=1).cpu().numpy()
        probs = torch.softmax(interest_scores, dim=1)
        entropy = (-(probs * (probs.clamp_min(1e-9).log())).sum(dim=1)).cpu().numpy()
        all_best_interest.extend(best_interest.tolist())
        all_interest_entropy.extend(entropy.tolist())
    counts = np.bincount(np.asarray(all_best_interest), minlength=model.n_interests).astype(np.float64)
    counts = counts / counts.sum()
    return {
        "interest_usage_entropy": float(-(counts * np.log(counts + 1e-12)).sum()),
        "interest_usage_max_share": float(counts.max()),
        "interest_target_score_entropy_mean": float(np.mean(all_interest_entropy)),
    }


class EarlyStopper:
    def __init__(self, patience: int = 5, min_delta: float = 1e-4, mode: str = "max"):
        assert mode in ("max", "min")
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.bad_epochs = 0
        self.best_state = None

    def _is_improvement(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        return value < self.best - self.min_delta

    def step(self, value: float, model: nn.Module) -> bool:
        if self._is_improvement(value):
            self.best = value
            self.bad_epochs = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return -torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores) + 1e-9).mean()


def diversity_regularization(z: torch.Tensor) -> torch.Tensor:
    z = F.normalize(z, dim=-1)
    sim = torch.matmul(z, z.transpose(1, 2))
    K = sim.size(1)
    eye = torch.eye(K, device=sim.device).unsqueeze(0)
    off_diag = sim * (1.0 - eye)
    return (off_diag ** 2).mean()


def label_aware_comirec_loss(
    model: ComiRecSAStyleSASRec,
    seq: torch.Tensor,
    pos: torch.Tensor,
    negs: torch.Tensor,
    label_tau: float,
    diversity_reg_weight: float,
    use_label_aware: bool = True,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    z = model.get_interest_repr(seq)  # [B, K, D]

    pos_item_vec = model.item_emb(pos)  # [B, D]
    neg_item_vec = model.item_emb(negs)  # [B, N, D]

    pos_scores_per_interest = torch.einsum("bkd,bd->bk", z, pos_item_vec)  # [B, K]
    neg_scores_per_interest = torch.einsum("bkd,bnd->bkn", z, neg_item_vec)  # [B, K, N]

    if use_label_aware:
        routing_weights = torch.softmax(pos_scores_per_interest / label_tau, dim=1)  # [B, K]
        pos_scores = (routing_weights * pos_scores_per_interest).sum(dim=1)  # [B]
        neg_scores = (routing_weights.unsqueeze(-1) * neg_scores_per_interest).sum(dim=1)  # [B, N]
    else:
        pos_scores = pos_scores_per_interest.max(dim=1).values
        neg_scores = neg_scores_per_interest.max(dim=1).values

    rec_loss = bpr_loss(pos_scores, neg_scores)
    div_loss = diversity_regularization(z)
    total_loss = rec_loss + diversity_reg_weight * div_loss

    aux = {
        "rec_loss": float(rec_loss.detach().item()),
        "div_loss": float(div_loss.detach().item()),
    }
    return total_loss, aux


def make_pairwise_loader(train_seqs, seed: int):
    ds = PairwiseNextItemDataset(
        train_seqs=train_seqs,
        max_len=MAX_SEQ_LEN,
        n_items=n_items,
        n_neg=50,
        max_prefix_samples_per_user=20,
        seed=seed,
    )
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
    return ds, dl

## 12. Evaluation helpers and subset reporting

This section contains:
- subset-based evaluation helpers,
- TopPop full-ranking evaluation,
- and utilities for building per-model subset result tables.

In [ ]:
def build_test_subset_rows_for_toppop(
    model_name: str,
    seed,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    pop_counter: Counter,
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
) -> List[Dict[str, object]]:
    ranked = predict_toppop_ranked(
        eval_cases=eval_cases,
        train_or_eval_seqs=eval_seqs,
        pop_counter=pop_counter,
        n_items=n_items,
        k=k,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )


def build_test_subset_rows_for_repr_model(
    model_name: str,
    seed,
    model: SequenceModelBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> List[Dict[str, object]]:
    ranked = predict_repr_model_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )


def build_test_subset_rows_for_multi_interest_model(
    model_name: str,
    seed,
    model: MultiInterestBase,
    eval_cases: EvalCases,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    subset_masks: Dict[str, np.ndarray],
    k: int = 20,
    eval_batch_size: int = 256,
    item_chunk_size: int = 4096,
) -> List[Dict[str, object]]:
    ranked = predict_multi_interest_ranked(
        model=model,
        eval_cases=eval_cases,
        eval_seqs=eval_seqs,
        n_items=n_items,
        k=k,
        eval_batch_size=eval_batch_size,
        item_chunk_size=item_chunk_size,
    )
    return summarize_subset_metrics_from_ranked(
        ranked=ranked,
        targets=eval_cases.targets,
        n_items=n_items,
        k=k,
        subset_masks=subset_masks,
        model_name=model_name,
        seed=seed,
    )

In [18]:
top_r, top_n, top_m, top_cov = evaluate_toppop_full_ranking(
    test_cases,
    test_seqs,
    pop,
    n_items,
    TOPK,
)
print(f"TopPop TEST: R@{TOPK}={top_r:.4f} N@{TOPK}={top_n:.4f} MRR@{TOPK}={top_m:.4f} Coverage@{TOPK}={top_cov:.4f}")

TopPop TEST: R@20=0.0184 N@20=0.0075 MRR@20=0.0045 Coverage@20=0.0061


## 13. Training loops for benchmark models

This section contains the training procedures for:
- GRU4Rec
- SASRec
- SASRec + CL4SRec-style SSL
- SASRec + SimDCL-style SSL
- FMLP-Rec-style efficient baseline
- ComiRec-SA-style SASRec
- ComiRec-SA-style SASRec with label-aware routing and diversity regularization

All models use the same validation protocol and early stopping logic.

In [ ]:
def train_gru4rec(train_seqs, valid_cases, test_cases, seed: int):
    set_all_seeds(seed)
    model = GRU4Rec(
        n_items=n_items,
        d_model=D_MODEL,
        hidden_size=GRU_HIDDEN,
        n_layers=1,
        dropout=DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for seq, pos, negs in dl:
            seq, pos, negs = seq.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)
            u = model.get_user_repr(seq)
            pos_scores = model.score_items_from_repr(u, pos.unsqueeze(1)).squeeze(1)
            neg_scores = model.score_items_from_repr(u, negs)
            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(f"[GRU4Rec][seed={seed}] epoch {ep+1}/{EPOCHS}: loss={total/max(1, len(dl)):.4f} | valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}")

        if stopper.step(n, model):
            print(f"[GRU4Rec][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name="GRU4Rec",
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows


def pretrain_ssl(model: SequenceModelBase, train_seqs, seed: int, ssl_mode: str):
    set_all_seeds(seed)
    ds_ssl = ContrastiveDataset(train_seqs, MAX_SEQ_LEN)
    dl_ssl = DataLoader(ds_ssl, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    for ep in range(SSL_EPOCHS):
        model.train()
        total = 0.0

        for batch in dl_ssl:
            if ssl_mode == "cl4srec":
                v1, v2 = batch
                v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
                z1 = model.pooled_ssl_repr(v1)
                z2 = model.pooled_ssl_repr(v2)
                loss = info_nce_loss(z1, z2, temperature=SSL_TAU)

            elif ssl_mode == "simdcl":
                v1, _ = batch
                v1 = v1.to(DEVICE)
                z = model.pooled_ssl_repr(v1)
                loss = simdcl_loss(z, noise_std=SIMDCL_NOISE_STD, temperature=SSL_TAU)
            else:
                raise ValueError(f"Unknown ssl_mode={ssl_mode}")

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        print(f"[SSL-{ssl_mode}] epoch {ep+1}/{SSL_EPOCHS}: loss={total/max(1, len(dl_ssl)):.4f}")


def train_sasrec(train_seqs, valid_cases, test_cases, seed: int, ssl_mode: Optional[str] = None):
    set_all_seeds(seed)
    model = SASRec(
        n_items=n_items,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    if ssl_mode is not None:
        print(f"[SASRec][seed={seed}] SSL pretraining mode={ssl_mode}")
        pretrain_ssl(model, train_seqs=train_seqs, seed=seed, ssl_mode=ssl_mode)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    tag = {
        None: "SASRec",
        "cl4srec": "SASRec+CL4SRec",
        "simdcl": "SASRec+SimDCL",
    }[ssl_mode]

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for seq, pos, negs in dl:
            seq, pos, negs = seq.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)
            u = model.get_user_repr(seq)
            pos_scores = model.score_items_from_repr(u, pos.unsqueeze(1)).squeeze(1)
            neg_scores = model.score_items_from_repr(u, negs)
            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(f"[{tag}][seed={seed}] epoch {ep+1}/{EPOCHS}: loss={total/max(1, len(dl)):.4f} | valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}")

        if stopper.step(n, model):
            print(f"[{tag}][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name=tag,
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows


def train_fmlprec(train_seqs, valid_cases, test_cases, seed: int):
    set_all_seeds(seed)
    model = FMLPRec(
        n_items=n_items,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_layers=FMLP_N_LAYERS,
        dropout=FMLP_DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    for ep in range(EPOCHS):
        model.train()
        total = 0.0

        for seq, pos, negs in dl:
            seq, pos, negs = seq.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)
            u = model.get_user_repr(seq)
            pos_scores = model.score_items_from_repr(u, pos.unsqueeze(1)).squeeze(1)
            neg_scores = model.score_items_from_repr(u, negs)
            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        r, n, m, cov = evaluate_repr_model_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(f"[FMLP-Rec][seed={seed}] epoch {ep+1}/{EPOCHS}: loss={total/max(1, len(dl)):.4f} | valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}")

        if stopper.step(n, model):
            print(f"[FMLP-Rec][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_repr_model_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
    }

    subset_rows = build_test_subset_rows_for_repr_model(
        model_name="FMLP-Rec",
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows


def train_comirec_sa_style(train_seqs, valid_cases, test_cases, seed: int, use_label_aware: bool):
    set_all_seeds(seed)
    model = ComiRecSAStyleSASRec(
        n_items=n_items,
        max_len=MAX_SEQ_LEN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
        n_interests=N_INTERESTS,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    _, dl = make_pairwise_loader(train_seqs, seed)
    stopper = EarlyStopper(patience=PATIENCE, min_delta=1e-4, mode="max")

    tag = (
        "ComiRecSA"
        if not use_label_aware
        else "ComiRecSA+LabelAware+DivReg"
    )

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        total_rec = 0.0
        total_div = 0.0

        for seq, pos, negs in dl:
            seq, pos, negs = seq.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)

            loss, aux = label_aware_comirec_loss(
                model=model,
                seq=seq,
                pos=pos,
                negs=negs,
                label_tau=COMIREC_LABEL_TAU,
                diversity_reg_weight=COMIREC_DIVERSITY_REG,
                use_label_aware=use_label_aware,
            )

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += float(loss.item())
            total_rec += aux["rec_loss"]
            total_div += aux["div_loss"]

        r, n, m, cov = evaluate_multi_interest_full_ranking(
            model, valid_cases, valid_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
        )
        print(
            f"[{tag}][seed={seed}] epoch {ep+1}/{EPOCHS}: "
            f"loss={total/max(1, len(dl)):.4f} "
            f"rec={total_rec/max(1, len(dl)):.4f} "
            f"div={total_div/max(1, len(dl)):.4f} | "
            f"valid R@{TOPK}={r:.4f} N@{TOPK}={n:.4f} MRR@{TOPK}={m:.4f} COV@{TOPK}={cov:.4f}"
        )

        if stopper.step(n, model):
            print(f"[{tag}][seed={seed}] early stopping at epoch {ep+1}.")
            break

    stopper.restore_best(model)

    r, n, m, cov = evaluate_multi_interest_full_ranking(
        model, test_cases, test_seqs, n_items, k=TOPK, eval_batch_size=EVAL_BATCH_SIZE
    )

    metrics = {
        "Recall@20": r,
        "NDCG@20": n,
        "MRR@20": m,
        "Coverage@20": cov,
        **analyze_interest_usage(model, test_cases),
    }

    subset_rows = build_test_subset_rows_for_multi_interest_model(
        model_name=tag,
        seed=seed,
        model=model,
        eval_cases=test_cases,
        eval_seqs=test_seqs,
        n_items=n_items,
        subset_masks=TEST_SUBSET_MASKS,
        k=TOPK,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
    return model, metrics, subset_rows

## 14. Main experiment run

Here I define the main experiment plan across:
- classic baselines,
- stronger SSL baselines,
- an efficient sequential alternative,
- and a sharper multi-interest comparison.

The main multi-interest comparison is now:
- ComiRec-SA baseline
- ComiRec-SA + label-aware routing + diversity regularization

This is the main benchmark block used to test whether a stronger multi-interest formulation helps especially on heterogeneous and long-tail playlist continuation.

In [ ]:
results = []
subset_results = []

experiment_plan = [
    ("TopPop", "fixed", lambda: (
        None,
        {
            "Recall@20": top_r,
            "NDCG@20": top_n,
            "MRR@20": top_m,
            "Coverage@20": top_cov,
        },
        build_test_subset_rows_for_toppop(
            model_name="TopPop",
            seed="fixed",
            eval_cases=test_cases,
            eval_seqs=test_seqs,
            pop_counter=pop,
            n_items=n_items,
            subset_masks=TEST_SUBSET_MASKS,
            k=TOPK,
        ),
    ))
]

for seed in SEEDS:
    experiment_plan.append(
        ("GRU4Rec", seed, lambda seed=seed: train_gru4rec(train_seqs, valid_cases, test_cases, seed))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("SASRec", seed, lambda seed=seed: train_sasrec(train_seqs, valid_cases, test_cases, seed, ssl_mode=None))
    )

if DO_CL4SREC_SSL:
    for seed in SEEDS:
        experiment_plan.append(
            ("SASRec+CL4SRec", seed, lambda seed=seed: train_sasrec(train_seqs, valid_cases, test_cases, seed, ssl_mode="cl4srec"))
        )

if DO_SIMDCL:
    for seed in SEEDS:
        experiment_plan.append(
            ("SASRec+SimDCL", seed, lambda seed=seed: train_sasrec(train_seqs, valid_cases, test_cases, seed, ssl_mode="simdcl"))
        )

for seed in SEEDS:
    experiment_plan.append(
        ("FMLP-Rec", seed, lambda seed=seed: train_fmlprec(train_seqs, valid_cases, test_cases, seed))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("ComiRecSA", seed, lambda seed=seed: train_comirec_sa_style(
            train_seqs, valid_cases, test_cases, seed, use_label_aware=False
        ))
    )

for seed in SEEDS:
    experiment_plan.append(
        ("ComiRecSA+LabelAware+DivReg", seed, lambda seed=seed: train_comirec_sa_style(
            train_seqs, valid_cases, test_cases, seed, use_label_aware=True
        ))
    )

print(f"Total runs: {len(experiment_plan)}")

for run_idx, (model_name, seed, run_fn) in enumerate(tqdm(experiment_plan, desc="Experiments"), start=1):
    print("=" * 80)
    print(f"START [{run_idx}/{len(experiment_plan)}] model={model_name}, seed={seed}")

    t0 = perf_counter()
    model_obj, metrics, subset_rows = run_fn()
    elapsed = perf_counter() - t0

    row = {
        "model": model_name,
        "seed": seed,
        "runtime_sec": round(elapsed, 2),
        **metrics,
    }
    results.append(row)
    subset_results.extend(subset_rows)

    results_df = pd.DataFrame(results)
    subset_results_df = pd.DataFrame(subset_results)

    results_df.to_csv(PROJECT_DIR / "results_full_ranking_checkpoint.csv", index=False)
    subset_results_df.to_csv(PROJECT_DIR / "results_subset_checkpoint.csv", index=False)

    print(f"DONE  [{run_idx}/{len(experiment_plan)}] model={model_name}, seed={seed}, time={elapsed:.2f}s")

    clear_output(wait=True)
    print(f"Completed runs: {run_idx}/{len(experiment_plan)}")
    display(results_df.sort_values(["model", "seed"], kind="stable").reset_index(drop=True))

results_df = pd.DataFrame(results).sort_values(["model", "seed"], kind="stable").reset_index(drop=True)
subset_results_df = pd.DataFrame(subset_results).sort_values(["model", "subset", "seed"], kind="stable").reset_index(drop=True)

display(results_df)
display(subset_results_df.head(30))

Completed runs: 2/11


,model,seed,runtime_sec,Recall@20,NDCG@20,MRR@20,Coverage@20,param_l2_norm,margin_mean,margin_std,margin_positive_rate,sharpness_drop_recall,sharpness_drop_ndcg,sharpness_drop_mrr
0,GRU4Rec,42,18987.59,0.206025,0.089824,0.057742,0.9823,3207.822246,-3.230559,2.177221,0.02933,0.000334,0.000165,0.000126
1,TopPop,fixed,4.04,0.018411,0.007489,0.004471,0.0061,NaN,NaN,NaN,NaN,NaN,NaN,NaN


START [3/11] model=GRU4Rec, seed=43
[GRU4Rec][seed=43] epoch 1/30: train_loss=0.1963 | valid R@20=0.1631 N@20=0.0660 MRR@20=0.0395 COV@20=0.7628


## 15. Aggregate results across seeds

After all runs finish, I summarize:
- full test metrics,
- subset metrics,
- and interest-usage statistics
across seeds with mean values and confidence intervals.

In [ ]:
def summarize_metric(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    n = len(x)
    mean = x.mean()
    std = x.std(ddof=1) if n > 1 else 0.0
    ci95 = 1.96 * std / math.sqrt(n) if n > 1 else 0.0
    return pd.Series({"mean": mean, "std": std, "ci95": ci95, "n_seeds": n})


main_metric_cols = ["Recall@20", "NDCG@20", "MRR@20", "Coverage@20"]
summary_rows = []
for model_name, g in results_df.groupby("model", sort=False):
    row = {"model": model_name}
    if model_name == "TopPop":
        for col in main_metric_cols:
            row[col] = f"{float(g[col].iloc[0]):.4f}"
        row["n_seeds"] = 1
    else:
        for col in main_metric_cols:
            s = summarize_metric(g[col])
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"
        row["n_seeds"] = int(summarize_metric(g["Recall@20"])["n_seeds"])
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
subset_summary_rows = []

for (model_name, subset_name), g in subset_results_df.groupby(["model", "subset"], sort=False):
    row = {
        "model": model_name,
        "subset": subset_name,
        "n_cases": int(g["n_cases"].iloc[0]),
    }

    for col in [f"Recall@{TOPK}", f"NDCG@{TOPK}", f"MRR@{TOPK}", f"Coverage@{TOPK}"]:
        vals = g[col].astype(float)
        if len(vals) == 1:
            row[col] = f"{float(vals.iloc[0]):.4f}"
        else:
            s = summarize_metric(vals)
            row[col] = f"{s['mean']:.4f} ± {s['ci95']:.4f}"

    subset_summary_rows.append(row)

subset_summary_df = pd.DataFrame(subset_summary_rows)
display(subset_summary_df.sort_values(["subset", "model"]).reset_index(drop=True))

In [ ]:
subset_recall_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"Recall@{TOPK}",
    aggfunc="mean",
)

subset_ndcg_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"NDCG@{TOPK}",
    aggfunc="mean",
)

subset_mrr_pivot = subset_results_df.pivot_table(
    index="subset",
    columns="model",
    values=f"MRR@{TOPK}",
    aggfunc="mean",
)

display(subset_recall_pivot)
display(subset_ndcg_pivot)
display(subset_mrr_pivot)